In [1]:
!pip install "stable-baselines3[extra]" "gymnasium[atari,accept-rom-license]" ale-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 12.0 MB/s eta 0:00:00


In [2]:
%%writefile train.py
"""
train.py - Train a DQN agent on Atari Pong using Stable Baselines3.

Can be used two ways:
  1. From the command line:
     python train.py --name exp02_high_lr --lr 0.001 --timesteps 150000
  2. Imported in a notebook:
     from train import train_dqn
     train_dqn(name="exp02_high_lr", lr=1e-3)

Each run saves the model as dqn_model_<name>.zip, logs to TensorBoard,
and appends a summary row (final evaluation reward, etc.) to results.csv.
"""

import argparse
import csv
import os
import time

from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import VecFrameStack

import ale_py
import gymnasium as gym
gym.register_envs(ale_py)   # registers the Atari environments with Gymnasium

ENV_ID = "ALE/Pong-v5"      # modern naming; old PongNoFrameskip-v4 was removed
RESULTS_FILE = "results.csv"


def make_env(seed=42):
    """Standard Atari setup: preprocessing + 4-frame stacking so the
    agent can perceive motion (one frame can't show ball direction)."""
    env = make_atari_env(
        ENV_ID, n_envs=1, seed=seed,
        env_kwargs={"frameskip": 1, "repeat_action_probability": 0.0},
    )
    env = VecFrameStack(env, n_stack=4)
    return env


def train_dqn(
    name,
    lr=1e-4,
    gamma=0.99,
    batch_size=32,
    eps_start=1.0,
    eps_end=0.05,
    exploration_fraction=0.1,   # SB3's "epsilon decay": fraction of training
                                # over which epsilon decays linearly.
                                # Smaller = faster decay.
    policy="CnnPolicy",
    timesteps=150_000,
    seed=42,
):
    """Train one DQN agent with the given hyperparameters, save the model,
    evaluate it greedily, and log a summary row to results.csv."""
    env = make_env(seed)

    model = DQN(
        policy=policy,
        env=env,
        learning_rate=lr,
        gamma=gamma,
        batch_size=batch_size,
        exploration_initial_eps=eps_start,
        exploration_final_eps=eps_end,
        exploration_fraction=exploration_fraction,
        buffer_size=50_000,        # replay memory (kept modest for Kaggle RAM)
        learning_starts=10_000,    # random play before learning begins
        train_freq=4,
        target_update_interval=1_000,
        verbose=1,
        tensorboard_log=f"./tb_logs/{name}",
        seed=seed,
    )

    print(f"\n{'='*60}")
    print(f"=== {name}: policy={policy}, lr={lr}, gamma={gamma}, "
          f"batch={batch_size}, eps={eps_start}->{eps_end} "
          f"over {exploration_fraction*100:.0f}% of training ===")
    print(f"{'='*60}\n")

    start = time.time()
    model.learn(total_timesteps=timesteps, log_interval=25)
    minutes = (time.time() - start) / 60

    model.save(f"dqn_model_{name}")

    # Greedy evaluation (deterministic=True = GreedyQPolicy): how good is
    # the agent when it stops exploring and just plays its best?
    eval_env = make_env(seed=123)
    mean_reward, std_reward = evaluate_policy(
        model, eval_env, n_eval_episodes=3, deterministic=True
    )
    eval_env.close()
    env.close()

    print(f"\n>>> {name} done in {minutes:.1f} min | "
          f"greedy eval reward: {mean_reward:.1f} +/- {std_reward:.1f}\n")

    # Append summary row to results.csv -> this becomes your table
    write_header = not os.path.exists(RESULTS_FILE)
    with open(RESULTS_FILE, "a", newline="") as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow([
                "experiment", "policy", "lr", "gamma", "batch_size",
                "eps_start", "eps_end", "exploration_fraction",
                "timesteps", "train_minutes", "eval_reward_mean",
                "eval_reward_std",
            ])
        writer.writerow([
            name, policy, lr, gamma, batch_size, eps_start, eps_end,
            exploration_fraction, timesteps, round(minutes, 1),
            round(mean_reward, 2), round(std_reward, 2),
        ])

    return mean_reward


def parse_args():
    p = argparse.ArgumentParser(description="Train a DQN agent on Atari Pong")
    p.add_argument("--name", required=True, help="experiment name, e.g. exp01_baseline")
    p.add_argument("--lr", type=float, default=1e-4)
    p.add_argument("--gamma", type=float, default=0.99)
    p.add_argument("--batch", type=int, default=32)
    p.add_argument("--eps_start", type=float, default=1.0)
    p.add_argument("--eps_end", type=float, default=0.05)
    p.add_argument("--exploration_fraction", type=float, default=0.1)
    p.add_argument("--policy", default="CnnPolicy", choices=["CnnPolicy", "MlpPolicy"])
    p.add_argument("--timesteps", type=int, default=150_000)
    return p.parse_args()


if __name__ == "__main__":
    args = parse_args()
    train_dqn(
        name=args.name,
        lr=args.lr,
        gamma=args.gamma,
        batch_size=args.batch,
        eps_start=args.eps_start,
        eps_end=args.eps_end,
        exploration_fraction=args.exploration_fraction,
        policy=args.policy,
        timesteps=args.timesteps,
    )

Writing train.py


In [3]:
import importlib
import train
importlib.reload(train)
from train import train_dqn

# train_dqn(name="smoke_test", timesteps=20_000)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
cd "/content/drive/MyDrive/formative3_backup"

/content/drive/MyDrive/formative3_backup


In [6]:
import pandas as pd

results = pd.read_csv(
    "/content/drive/MyDrive/formative3_backup/results.csv"
)

print(results)

              experiment     policy       lr  gamma  batch_size  eps_start  \
0    d01_highlr_bigbatch  CnnPolicy  0.00100   0.99         128        1.0   
1    d02_lowlr_slowdecay  CnnPolicy  0.00001   0.99          32        1.0   
2  d03_batch128_gamma095  CnnPolicy  0.00010   0.95         128        1.0   
3   d04_fastdecay_lowend  CnnPolicy  0.00010   0.99          32        1.0   
4  d05_batch64_slowdecay  CnnPolicy  0.00010   0.99          64        1.0   
5      d06_lr5e4_batch64  CnnPolicy  0.00050   0.99          64        1.0   

   eps_end  exploration_fraction  timesteps  train_minutes  eval_reward_mean  \
0     0.05                  0.10     500000           35.3            -21.00   
1     0.05                  0.30     500000           28.7            -20.33   
2     0.05                  0.10     500000           35.0             -1.00   
3     0.01                  0.02     500000           30.4            -19.33   
4     0.05                  0.30     500000          

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [7]:
import importlib
import train
importlib.reload(train)
from train import train_dqn

EXPERIMENTS = [
   dict(name="d07_highgamma_batch128", gamma=0.999, batch_size=128),
    dict(name="d08_low_eps_start", eps_start=0.5),
    dict(name="d09_lowgamma_fastdecay", gamma=0.90, exploration_fraction=0.02),
    dict(name="d10_combo_best", lr=1e-4, batch_size=128, exploration_fraction=0.3, eps_end=0.01),
]

for config in EXPERIMENTS:
    train_dqn(timesteps=500_000, **config)

Using cpu device
Wrapping the env in a VecTransposeImage.

=== d07_highgamma_batch128: policy=CnnPolicy, lr=0.0001, gamma=0.999, batch=128, eps=1.0->0.05 over 10% of training ===

Logging to ./tb_logs/d07_highgamma_batch128/DQN_2


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


KeyboardInterrupt: 

In [ ]:
#import importlib
#import train
#importlib.reload(train)
#from train import train_dqn

# best combo, trained longer; this becomes the final dqn_model.zip
#train_dqn(name="exp10_best", batch_size=128, exploration_fraction=0.2,
 #         lr=1e-4, gamma=0.99, timesteps=1_000_000)

# MLP comparison for the CNN vs MLP discussion (short run is enough)
#train_dqn(name="exp_mlp", policy="MlpPolicy", timesteps=150_000)

In [ ]:
import pandas as pd
df = pd.read_csv("results.csv")
df.sort_values("eval_reward_mean", ascending=False)